# Model Comparison & Statistical Tests

**Course:** Học Thống Kê / Data Science Capstone  
**Purpose:** Compare **DistilBERT** vs **TF-IDF baselines** (+ optional **LoRA**) on the **same held-out test split**  
**Sources:**  
- `artifacts/results/evaluation.json` (`make evaluate`, `make hypothesis-tests`)  
- `artifacts/results/lora_comparison.json` (`make lora-quick`, optional)  

---

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
GOLD = '#e8c547'
BERT_COLOR = '#3498db'
BASE_COLORS = ['#95a5a6', '#e67e22', '#1abc9c', '#9b59b6']

ROOT = Path('..').resolve()
RESULTS = ROOT / 'artifacts' / 'results'
EVAL_JSON = RESULTS / 'evaluation.json'
LORA_JSON = RESULTS / 'lora_comparison.json'
RESULTS.mkdir(parents=True, exist_ok=True)

if not EVAL_JSON.is_file():
    raise FileNotFoundError(f'Missing {EVAL_JSON}. Run: make train && make evaluate && make baseline')

artifact = json.loads(EVAL_JSON.read_text(encoding='utf-8'))
test_m = artifact.get('splits', {}).get('test', {}).get('metrics', {})
test_ci = test_m.get('confidence_intervals', {})
baselines = artifact.get('baselines', {})
hyp = artifact.get('hypothesis_tests', {}).get('test', {})

print('DistilBERT test F1:', round(test_m.get('f1', 0), 4))
print('Baselines:', list(baselines.keys()))
print('Hypothesis tests present:', bool(hyp))

In [ ]:
# ── Build comparison table (test split) ───────────────────────────
rows = [{
    'Model': 'DistilBERT (fine-tuned)',
    'Family': 'Transformer',
    'Accuracy': test_m.get('accuracy'),
    'F1': test_m.get('f1'),
    'Precision': test_m.get('precision'),
    'Recall': test_m.get('recall'),
    'ROC-AUC': test_m.get('roc_auc'),
}]

for key, bl in baselines.items():
    tm = bl.get('splits', {}).get('test', {}).get('metrics', {})
    rows.append({
        'Model': bl.get('model', key),
        'Family': 'TF-IDF',
        'Accuracy': tm.get('accuracy'),
        'F1': tm.get('f1'),
        'Precision': tm.get('precision'),
        'Recall': tm.get('recall'),
        'ROC-AUC': tm.get('roc_auc'),
    })

if LORA_JSON.is_file():
    lora = json.loads(LORA_JSON.read_text(encoding='utf-8'))
    lm = lora.get('test_metrics_lora', {})
    rows.append({
        'Model': 'DistilBERT + LoRA (PEFT)',
        'Family': 'LoRA adapter',
        'Accuracy': lm.get('accuracy'),
        'F1': lm.get('f1'),
        'Precision': lm.get('precision'),
        'Recall': lm.get('recall'),
        'ROC-AUC': lm.get('roc_auc'),
    })
else:
    print('ℹ  No lora_comparison.json — run: make lora-quick (optional)')

cmp = pd.DataFrame(rows)
cmp_display = cmp.copy()
for c in ['Accuracy', 'F1', 'Precision', 'Recall', 'ROC-AUC']:
    cmp_display[c] = cmp_display[c].apply(lambda x: f'{x*100:.2f}%' if pd.notna(x) else '—')
cmp_display

In [ ]:
# ── Bar chart: F1 & Accuracy on test ──────────────────────────────
plot_df = cmp.dropna(subset=['F1']).copy()
x = np.arange(len(plot_df))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
colors = [BERT_COLOR if 'DistilBERT' in m and 'LoRA' not in m else GOLD if 'LoRA' in m else BASE_COLORS[i % len(BASE_COLORS)]
          for i, m in enumerate(plot_df['Model'])]

ax.bar(x - w/2, plot_df['Accuracy'] * 100, w, label='Accuracy', color=colors, alpha=0.85, edgecolor='white')
ax.bar(x + w/2, plot_df['F1'] * 100, w, label='F1', color=colors, alpha=0.55, edgecolor='white', hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['Model'], rotation=25, ha='right')
ax.set_ylabel('Score (%)')
ax.set_ylim(85, 100)
ax.set_title('Test-Set Performance Comparison (τ = 0.5)', fontsize=14, color=GOLD)
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS / 'model_comparison_bars.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Delta vs primary TF-IDF logistic baseline ─────────────────────
bl_key = 'tfidf_logistic'
if bl_key in baselines:
    bl_f1 = baselines[bl_key]['splits']['test']['metrics'].get('f1', 0)
    bl_acc = baselines[bl_key]['splits']['test']['metrics'].get('accuracy', 0)
    deltas = pd.DataFrame({
        'Metric': ['Accuracy', 'F1', 'Precision', 'Recall'],
        'DistilBERT': [test_m.get('accuracy'), test_m.get('f1'), test_m.get('precision'), test_m.get('recall')],
        'TF-IDF LogReg': [bl_acc, bl_f1,
            baselines[bl_key]['splits']['test']['metrics'].get('precision'),
            baselines[bl_key]['splits']['test']['metrics'].get('recall')],
    })
    deltas['Δ (DistilBERT − baseline)'] = deltas['DistilBERT'] - deltas['TF-IDF LogReg']
    deltas['Δ (pp)'] = (deltas['Δ (DistilBERT − baseline)'] * 100).round(2)
    display_df = deltas.copy()
    for c in ['DistilBERT', 'TF-IDF LogReg', 'Δ (DistilBERT − baseline)']:
        display_df[c] = display_df[c].apply(lambda x: f'{x*100:.2f}%')
    display_df
else:
    print('Run make baseline first')

In [ ]:
# ── Bootstrap 95% CI (DistilBERT test) ────────────────────────────
ci_rows = []
for metric in ('accuracy', 'f1', 'precision', 'recall'):
    pt = test_m.get(metric)
    ci = test_ci.get(metric, {})
    if pt is None:
        continue
    ci_rows.append({
        'Metric': metric.upper(),
        'Point': f'{pt*100:.2f}%',
        'CI low': f"{ci.get('low', 0)*100:.2f}%" if ci.get('low') is not None else '—',
        'CI high': f"{ci.get('high', 0)*100:.2f}%" if ci.get('high') is not None else '—',
    })
ci_df = pd.DataFrame(ci_rows)
print('Bootstrap: 500 resamples, percentile method (see METHODOLOGY.md)')
ci_df.style.set_caption('DistilBERT — Test split 95% confidence intervals')

In [ ]:
# ── Hypothesis tests (McNemar + bootstrap difference) ───────────
if hyp:
    mcn = hyp.get('mcnemar', {})
    boot = hyp.get('bootstrap_difference', {})
    print('McNemar test (paired errors vs TF-IDF logistic)')
    print(f"  b = {mcn.get('discordant_b')}, c = {mcn.get('discordant_c')}")
    print(f"  p-value = {mcn.get('p_value', 0):.4f}")
    sig = mcn.get('significant_005')
    print(f"  Significant at α=0.05? {'Yes' if sig else 'No — report honestly'}")
    if mcn.get('interpretation'):
        print(f"  → {mcn['interpretation']}")
    print()
    for metric in ('accuracy', 'f1'):
        b = boot.get(metric, {})
        if not b:
            continue
        print(f"Bootstrap Δ{metric}: mean={b.get('mean_diff', 0):+.4f}  "
              f"CI=[{b.get('ci_low', 0):.4f}, {b.get('ci_high', 0):.4f}]  "
              f"p={b.get('p_value_two_sided', 0):.4f}")
else:
    print('No hypothesis_tests in evaluation.json — run: make hypothesis-tests')

In [ ]:
# ── Confusion matrix (DistilBERT test, from metrics if available) ─
cm = test_m.get('confusion_matrix')
if cm and len(cm) == 2:
    mat = np.array([[cm[0][0], cm[0][1]], [cm[1][0], cm[1][1]]])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(mat, annot=True, fmt=',d', cmap='YlOrRd', ax=ax,
                xticklabels=['Pred Rotten', 'Pred Fresh'],
                yticklabels=['True Rotten', 'True Fresh'],
                cbar_kws={'label': 'Count'})
    ax.set_title('DistilBERT Confusion Matrix (Test)', fontsize=14, color=GOLD)
    plt.tight_layout()
    fig.savefig(RESULTS / 'model_comparison_cm.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
else:
    print('Confusion matrix not embedded in JSON — see evaluation.html or summary.html')

In [ ]:
# ── Export LaTeX-ready table for báo cáo ────────────────────────
export = cmp[['Model', 'Accuracy', 'F1', 'Precision', 'Recall']].copy()
for c in ['Accuracy', 'F1', 'Precision', 'Recall']:
    export[c] = (export[c] * 100).round(2)
tex_path = RESULTS / 'model_comparison_table.csv'
export.to_csv(tex_path, index=False)
print(f'Wrote {tex_path}')

---

## Statistical interpretation (for báo cáo)

| Question | Where to answer |
|----------|-----------------|
| Is DistilBERT better than TF-IDF on test F1? | ΔF1 table above + `summary.html` |
| Is the improvement **significant**? | McNemar p-value + bootstrap Δ p-value |
| Uncertainty on point estimates? | Bootstrap 95% CI table |
| Why keep DistilBERT if not significant? | `docs/DEFENSE_SLIDE_LIMITATIONS.md` |

### Key principle

> Report **test-split** metrics only. Validation is for monitoring; test is touched once for final evidence.

**Figures:** `model_comparison_bars.png`, `model_comparison_cm.png`  
**Pipeline:** `make capstone` regenerates all artifacts this notebook reads.

---